# Buddy Voice Clone — Google Colab Android Backend

Free/open Chatterbox backend for Little Red’s Big Studio. Run this notebook from Android in Google Colab. It installs the current open **Chatterbox Multilingual V3**, detects the assigned GPU, starts a private token-protected HTTP server, creates a temporary Cloudflare Quick Tunnel, accepts Android voice uploads plus transcript fields, and returns a real WAV clone.

**Important:** Google Colab free GPU access is subject to availability/quota. Keep this notebook running while Buddy needs the backend. The temporary tunnel URL changes when the runtime restarts.

Model source: Resemble AI Chatterbox. No preset voice is used: every clone request requires reference audio.

In [ ]:
import os, sys, subprocess, time, secrets, pathlib, json, urllib.request
print('Python:', sys.version)
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'No NVIDIA GPU detected. The server can fall back to CPU, but cloning will be much slower.')


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -U chatterbox-tts==0.1.7 fastapi uvicorn python-multipart soundfile
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('Dependencies ready.')


## Download the Buddy server
The server code lives in the Little Red’s Big Studio repository so the app and Colab backend stay in sync.

In [ ]:
RAW = 'https://raw.githubusercontent.com/GiggleLootCoin/little-reds-big-studio-f36b7ec4/main/notebooks/buddy_chatterbox_colab_server.py'
SERVER = '/content/buddy_chatterbox_colab_server.py'
urllib.request.urlretrieve(RAW, SERVER)
TOKEN = secrets.token_urlsafe(32)
os.environ['BUDDY_CHATTERBOX_TOKEN'] = TOKEN
os.environ['BUDDY_CHATTERBOX_PORT'] = '8080'
print('Backend token generated. Keep it private; it is required by Buddy.')


## Start Chatterbox
This loads the current Multilingual V3 checkpoint once and keeps it in GPU memory for subsequent clone requests.

In [ ]:
import subprocess, time, urllib.request
server_proc = subprocess.Popen([sys.executable, SERVER], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
print('Starting Chatterbox. First startup downloads the model and may take a few minutes...')
deadline = time.time() + 420
ready = False
while time.time() < deadline:
    if server_proc.poll() is not None:
        raise RuntimeError('Chatterbox server exited during startup. Check the server output above/below.')
    try:
        req = urllib.request.Request('http://127.0.0.1:8080/health', headers={'Authorization': f'Bearer {TOKEN}'})
        with urllib.request.urlopen(req, timeout=5) as r:
            health = json.loads(r.read())
        if health.get('ok'):
            print(json.dumps(health, indent=2))
            ready = True
            break
    except Exception:
        time.sleep(3)
if not ready:
    raise TimeoutError('Chatterbox did not become ready within 7 minutes. Check GPU availability and server output.')


## Create the temporary secure endpoint
Cloudflare Quick Tunnel gives this Colab runtime a temporary HTTPS URL. The URL is random and the API additionally requires the private Bearer token. Do not post the token publicly.

In [ ]:
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8080', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
public_url = None
deadline = time.time() + 60
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        if tunnel.poll() is not None: break
        time.sleep(1); continue
    print(line.rstrip())
    if 'trycloudflare.com' in line:
        import re
        m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0)
            break
if not public_url:
    raise RuntimeError('Could not obtain a Cloudflare Quick Tunnel URL.')
print('\n=== BUDDY COLAB BACKEND ===')
print('URL:   ', public_url)
print('TOKEN: ', TOKEN)
print('VOICE:  POST ' + public_url + '/v1/voice-clone')
print('HEALTH: GET  ' + public_url + '/health')
print('Keep this cell running. Stop/restart the Colab runtime and the URL/token will change.')


## Real upload smoke test
Run this on Android, choose a real authorized voice sample, and enter the exact transcript if you have one. The server accepts WAV/MP3/M4A/AAC/OGG/Opus/WebM/FLAC and other formats that FFmpeg can decode, then normalizes the reference to mono 24 kHz WAV before cloning.

In [ ]:
from google.colab import files
import requests, os
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No reference audio was selected.')
name, data = next(iter(uploaded.items()))
target_text = 'Hello. This is your cloned voice sample. Would you like to use this voice for Buddy now, or would you like to record again?'
reference_transcript = ''
print('Uploading:', name, 'bytes:', len(data))
response = requests.post(public_url + '/v1/voice-clone', headers={'Authorization': f'Bearer {TOKEN}'}, files={'audio': (name, data)}, data={'text': target_text, 'refText': reference_transcript, 'language': 'en', 'exaggeration': '0.5', 'cfg_weight': '0.5', 'temperature': '0.8'}, timeout=300)
print('HTTP:', response.status_code)
if response.status_code != 200:
    try: print(json.dumps(response.json(), indent=2))
    except Exception: print(response.text[:2000])
    raise RuntimeError('Clone smoke test failed.')
if not response.content.startswith(b'RIFF') or b'WAVE' not in response.content[:32]:
    raise RuntimeError('Server returned something other than a WAV artifact.')
artifact = '/content/buddy-cloned-live-test.wav'
open(artifact, 'wb').write(response.content)
print('REAL CLONED AUDIO VERIFIED:', artifact, 'bytes:', os.path.getsize(artifact))
files.download(artifact)


## Connect Buddy
Set these two Cloudflare Worker secrets/variables from the values printed above:

`COLAB_CHATTERBOX_URL` = the temporary `https://....trycloudflare.com` URL

`COLAB_CHATTERBOX_TOKEN` = the generated token

The deployed Worker will keep Hugging Face as the first route when it has capacity. A Hugging Face 402/429/503 sends the same uploaded reference audio to this Colab backend. The Worker does not silently switch to a preset voice.

In [ ]:
print('COLAB_CHATTERBOX_URL =', public_url)
print('COLAB_CHATTERBOX_TOKEN =', TOKEN)
print('\nDo not share the token. If you restart Colab, replace both Worker values with the new URL/token.')
